# 8주차 — Plant 팔레트: 6-DoF 쿼드로터

**후반부 시작.** 지금까지는 엔진을 만들었다 — 이제 정식 loopsym 의
팔레트를 신호사슬 순서로 하나씩 연다. 학기 초에 그린 다이어그램이
학기 말(실기체)까지 같은 그림이다.

```
[Sensors] → [Estimation] → [Navigation] → [Control] → [Allocation] → [Plant]
                                                                       ▲ 이번 주
```

목표: 13-상태 강체 동역학의 구조. **개루프 불안정 체험.**

## 이론 요약
- 상태 13개: 쿼터니언 q(4) + 각속도 ω(3) + 위치 NED(3) + 속도(3)
- 왜 쿼터니언인가: 오일러각은 ±90° 특이점 (짐벌락)
- 추력: 모터 4개가 body −z 로 민다. 토크: 모터 위치×추력 + 반작용(요)
- **개루프 = 제어 없음**: 쿼드로터는 자체 안정성이 없다 — 손을 놓으면 뒤집힌다
- fidelity 개념 (ARCHITECTURE §2): 오늘 이 블록의 드롭다운이 학기 말에
  Gazebo/실기체가 된다. **다이어그램은 하나.**

In [ ]:
# 뷰어 열기 — 뼈대 모델을 브라우저에서 편집한다.
# 완성하면 뷰어의 '저장' 으로 week08_model.json 을 **덮어쓰기** (제출물!)
import sys, webbrowser; sys.path.insert(0, '../..')
from loopsym import load_json, render_html
p = render_html(load_json('week08_model.json'), 'week08_viewer.html',
                title='8주차', t_end=8.0, dt=0.004)
webbrowser.open('file://' + p)

## 실습 — 호버 추력 손 계산
★hover_cmd 상수가 0.0 이라 기체가 자유낙하한다. 힘 평형 `m·g = N·k_F·u`
에서 u 를 계산하라 (m=1kg, g=9.81, N=4, k_F=m·g/2 — QuadPlant 기본값).
뷰어에서 ★hover_cmd 더블클릭 → value 수정 → 실행 → 고도 유지 확인 → 저장.

In [ ]:
# 노트북에서 바로 실행해 보기 (뷰어 없이) — 저장한 모델을 로드해 시뮬
import sys; sys.path.insert(0, '../..')
from loopsym import load_json, Simulator
bd = load_json('week08_model.json')
sim = Simulator(bd); sim.run(3.0, 0.004)
import matplotlib.pyplot as plt
for b in sim.bd.blocks:
    if type(b).__name__ == 'Scope':
        plt.figure(figsize=(8,2.6)); plt.title(b.name); plt.grid(True)
        for i, lab in enumerate(b.labels): plt.plot(b.t, b.data[i], label=lab)
        plt.legend()

## 과제
1. week08_model.json 완성 (채점기: 3초 뒤 고도 유지)
2. 리포트: u 를 ±2% 만 틀리면 몇 초 만에 1m 를 벗어나는가? 실험으로 확인하고
   '개루프는 왜 실용이 안 되는가' 를 한 문단으로.

In [ ]:
# ✅ 채점 — 전부 초록색이 되면 이번 주 완료!
!python -m pytest test_week08.py -q